In [1]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

I0000 00:00:1789117895.588031   24974 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789117895.621936   24974 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789117896.632459   24974 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.layers.Flatten(input_shape=(28, 28)))
    
    # Tune the number of units in the first Dense layer
    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    model.add(keras.layers.Dense(units=hp_units, activation='relu'))
    
    # Tune the learning rate for the optimizer
    hp_learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')

    model.add(keras.layers.Dense(10, activation='softmax'))
    
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

In [3]:
# load the dataset
fashion_mnist = keras.datasets.fashion_mnist
(x_train,y_train),(x_val,y_val) = fashion_mnist.load_data()

# Normalize the data
x_train = x_train / 255.0
x_val = x_val / 255.0

In [4]:
tuner = kt.Hyperband(build_model,
                     objective='val_accuracy',
                     max_epochs=10,
                     factor=3,
                     directory='my_dir',
                     project_name='hyperparam_tuning')

I0000 00:00:1789118245.075859   24974 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 753 MB memory:  -> device: 0, name: NVIDIA T1000, pci bus id: 0000:01:00.0, compute capability: 7.5
/home/bhoopender/tensorflow/tensorflow/lib/python3.10/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [5]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

In [6]:
tuner.search(x_train, y_train, epochs=10, validation_data=(x_val, y_val), callbacks=[stop_early])

Trial 30 Complete [00h 00m 23s]
val_accuracy: 0.8859000205993652

Best val_accuracy So Far: 0.8909000158309937
Total elapsed time: 00h 05m 07s


In [7]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal number of units is {best_hps.get('units')},
and the optimal learning rate for the optimizer is {best_hps.get('learning_rate')}.
""")


The optimal number of units is 288,
and the optimal learning rate for the optimizer is 0.000688702391905252.



In [8]:
# build the model with the optimal hyperparameters

model = tuner.hypermodel.build(best_hps)

# train the model
history = model.fit(x_train, y_train, epochs=10, validation_data=(x_val, y_val))

/home/bhoopender/tensorflow/tensorflow/lib/python3.10/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10


I0000 00:00:1789118918.491402   25275 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1092283__.12


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8281 - loss: 0.4885 - val_accuracy: 0.8365 - val_loss: 0.4435
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8666 - loss: 0.3671 - val_accuracy: 0.8600 - val_loss: 0.3897
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8812 - loss: 0.3268 - val_accuracy: 0.8589 - val_loss: 0.3798
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8894 - loss: 0.3012 - val_accuracy: 0.8724 - val_loss: 0.3521
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8963 - loss: 0.2836 - val_accuracy: 0.8801 - val_loss: 0.3300
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9003 - loss: 0.2681 - val_accuracy: 0.8731 - val_loss: 0.3477
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9058 - loss: 0.2538 - val_accuracy: 0.8869 - val_loss: 0.3233
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9099 - loss: 0.2430 - val_accurac

In [9]:
eval_result = model.evaluate(x_val, y_val)
print(f"\nTest loss: {eval_result[0]}, Test accuracy: {eval_result[1]}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 956us/step - accuracy: 0.8806 - loss: 0.3350

Test loss: 0.3350001573562622, Test accuracy: 0.8805999755859375
